In [6]:
import json

with open("/home/hyang/mediQ/src/results/scope_rawtest.jsonl") as f:
    records = [json.loads(l) for l in f]

for r in records:
    s = r["interactive_system"]
    info = r["info"]
    print(f"=== ID {r['id']} | GT: {info['correct_answer_idx']} | Pred: {s['letter_choice']} | {'CORRECT' if s['correct'] else 'WRONG'} ===")
    print(f"Context: {' '.join(info['context'])}")
    print(f"Question: {info['question']}")
    print(f"Options: {info['options']}")
    print()
    for i, (q, a) in enumerate(zip(s["questions"], s["answers"])):
        raw = s["temp_additional_info"][i].get("raw_action", "(none)")
        print(f"  Turn {i+1} raw_action : {raw}")
        print(f"  Turn {i+1} Q          : {q}")
        print(f"  Turn {i+1} A          : {a}")
        print()
    final_raw = s["temp_additional_info"][len(s["questions"])].get("raw_action", "(none)")
    print(f"  Final raw_action : {final_raw}")
    print(f"  Final decision   : {s['letter_choice']}")
    print()


=== ID 0 | GT: C | Pred: C | CORRECT ===
Context: A 21-year-old sexually active male complains of fever, pain during urination, and inflammation and pain in the right knee. A culture of the joint fluid shows a bacteria that does not ferment maltose and has no polysaccharide capsule. The physician orders antibiotic therapy for the patient.
Question: The mechanism of action of the medication given blocks cell wall synthesis, which of the following was given?
Options: {'A': 'Gentamicin', 'B': 'Ciprofloxacin', 'C': 'Ceftriaxone', 'D': 'Trimethoprim'}

  Turn 1 raw_action : Was the medication given orally or intravenously?
  Turn 1 Q          : Was the medication given orally or intravenously?
  Turn 1 A          : The patient cannot answer this question, please do not ask this question again.

  Final raw_action : FINAL ANSWER: C
  Final decision   : C

=== ID 1 | GT: A | Pred: A | CORRECT ===
Context: A 5-year-old girl is brought to the emergency department by her mother because of multip

In [7]:
# pip install accelerate
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import requests
import torch

model_id = "google/medgemma-4b-it"

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)

# Image attribution: Stillwaterising, CC0, via Wikimedia Commons
image_url = "https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png"
image = Image.open(requests.get(image_url, headers={"User-Agent": "example"}, stream=True).raw)

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are an expert radiologist."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe this X-ray"},
            {"type": "image", "image": image}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)


/home/hyang/miniconda3/envs/scope/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 883/883 [00:00<00:00, 1166.24it/s]


Okay, based on the provided chest X-ray image, here's a description:

**Overall Impression:**

The image shows a normal chest X-ray. The heart size appears within normal limits. The lungs are clear, with no obvious consolidation, effusions, or masses. The mediastinum is unremarkable. The bony structures of the rib cage and clavicles are intact.

**Specific Findings:**

*   **Heart:** The heart size appears to be within normal limits. The cardiothoracic ratio (the ratio of the heart's width to the chest's width) is likely less than 0.5, which is generally considered normal.
*   **Lungs:** The lungs are clear bilaterally. There are no obvious signs of pneumonia (consolidation), pleural effusion (fluid in the pleural space), or pneumothorax (air in the pleural space).
*   **Mediastinum:** The mediastinum (the space between the lungs containing the heart,
